# Customer Churn Prediction — Streamlit Ready
**PSEB Artificial Intelligence Bootcamp | Portfolio Project**

## Business Problem
Predict whether a telecom customer is likely to churn. The project prioritizes **recall for the churn class** because missing a customer who is about to leave is more costly than contacting an extra customer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = 'Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.info())
print('Duplicates:', df.duplicated().sum())
print(df.describe(include='all').T)

## Data Cleaning

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges:', df['TotalCharges'].isna().sum())
df = df.dropna(subset=['TotalCharges']).drop(columns=['customerID'], errors='ignore').copy()
print('Clean shape:', df.shape)

In [ ]:
churn_pct = df['Churn'].value_counts(normalize=True).mul(100)
print(churn_pct)
churn_pct.plot(kind='bar', title='Customer Churn Distribution')
plt.xticks(rotation=0); plt.ylabel('Percentage'); plt.show()

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)
print(contract_churn)
contract_churn.plot(kind='bar', title='Churn Rate by Contract')
plt.ylabel('Churn Rate (%)'); plt.xticks(rotation=0); plt.show()

In [ ]:
df['TenureGroup'] = pd.cut(df['tenure'], [0,6,12,24,48,72], labels=['0-6','7-12','13-24','25-48','49-72'], include_lowest=True)
tenure_churn = df.groupby('TenureGroup', observed=True)['Churn'].apply(lambda x:(x=='Yes').mean()*100)
print(tenure_churn)
tenure_churn.plot(kind='bar', title='Churn Rate by Tenure'); plt.ylabel('Churn Rate (%)'); plt.show()

## Machine Learning Preparation

In [ ]:
X = df.drop(columns=['Churn','TenureGroup'])
y = df['Churn']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, classification_report, roc_auc_score

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()
preprocessor = ColumnTransformer([('num',StandardScaler(),numeric_features),('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)])

In [ ]:
models = {
'Logistic Regression': Pipeline([('preprocessor',preprocessor),('model',LogisticRegression(class_weight='balanced',max_iter=2000,solver='liblinear'))]),
'Decision Tree': Pipeline([('preprocessor',preprocessor),('model',DecisionTreeClassifier(class_weight='balanced',random_state=42))]),
'Random Forest': Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(class_weight='balanced',random_state=42,n_estimators=300,n_jobs=-1))])}
rows=[]
for name, model in models.items():
    model.fit(X_train,y_train); pred=model.predict(X_test)
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Recall':recall_score(y_test,pred,pos_label='Yes')})
pd.DataFrame(rows).sort_values('Recall',ascending=False)

## Tune the Final Logistic Regression Model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
recall_scorer = make_scorer(recall_score, pos_label='Yes')
base = models['Logistic Regression']
param_grid = {'model__C':[0.01,0.1,1,10,100]}
grid = GridSearchCV(base,param_grid,cv=5,scoring=recall_scorer,n_jobs=-1)
grid.fit(X_train,y_train)
best_model = grid.best_estimator_
print('Best parameters:',grid.best_params_)
print('Best CV recall:',grid.best_score_)

In [ ]:
pred = best_model.predict(X_test)
prob = best_model.predict_proba(X_test)[:,list(best_model.classes_).index('Yes')]
print('Accuracy:',accuracy_score(y_test,pred))
print('Recall:',recall_score(y_test,pred,pos_label='Yes'))
print('ROC-AUC:',roc_auc_score((y_test=='Yes').astype(int),prob))
print(classification_report(y_test,pred,target_names=['Stayed','Churned']))

In [ ]:
import joblib, json
joblib.dump(best_model,'churn_model.joblib')
print('Saved churn_model.joblib')

## Streamlit Deployment Files
Keep these files together when deploying: `app.py`, `churn_model.joblib`, and `requirements.txt`. The dataset is included for reproducibility but the Streamlit app does not need to load it at runtime.

In [ ]:
# Verify the model can be loaded and used exactly as Streamlit will use it
loaded = joblib.load('churn_model.joblib')
example = X_test.head(1)
print('Prediction:', loaded.predict(example)[0])
print('Classes:', loaded.classes_)
print('Model verification successful.')